# A4 — Remediation Family Clustering (HDBSCAN)

The L2 catalog holds 3,870 individual checks, but an engineer does not remediate 3,870 different ways — "enable encryption at rest" is *one action* recurring across dozens of services and all three tools. A4 recovers those **remediation families**, so later studies can prioritise at the level of the action a team actually takes rather than the individual finding.

Logic lives in `src/misconfig/clustering.py` (CLAUDE.md rule 9).

**Method:** embed remediation-oriented text → UMAP → HDBSCAN. HDBSCAN is used rather than k-means because it finds variable-density clusters and does not force every check into a family — genuinely unique checks are allowed to stay as noise.

**Two parameters are swept rather than assumed** (CLAUDE.md rule 5):
- `min_cluster_size`, the main structural knob;
- the random seed — UMAP is stochastic, so stability across 10 seeds is measured with the Adjusted Rand Index and reported as mean ± std.

**Caveat:** only 46% of checks ship remediation text (Prowler mostly). For the rest the family is inferred from title and description, so families are closer to "what the check is about" than "the exact fix" wherever remediation text is missing.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.misconfig.clustering import run, log_stats

In [ ]:
results = run()
log_stats(results)

## The families

Each family is named by its most distinctive terms (mean TF-IDF over its members). A family spanning several tools or providers is the interesting case: it means the same remediation action recurs across vendors.

In [ ]:
results["families"].head(25)

## Stability

UMAP is stochastic, so a single run's families could be an artefact. Pairwise ARI across seeds says how reproducible the structure actually is — read this before treating any individual family as real.

In [ ]:
print(results["stability"])
results["seed_summary"]

## Sanity-check a family by reading it

A clustering is only useful if its groups are coherent to a human. Pick the largest families and read their members.

In [ ]:
assignments = results["assignments"]
for family in results["families"]["family"].head(3):
    members = assignments[assignments.family == family]
    print(f"--- family {family} ({len(members)} checks): {members.family_terms.iloc[0]} ---")
    for title in members.title.head(8):
        print("   ", title)
    print()